# Transformaciones lineales: la matemática detrás de rotar y espejar

**Material complementario — Laboratorio de Tecnologías de la Imagen Digital**

**IFTS 24 - 2026**

---

> **Ubicación sugerida:** este cuaderno complementa el encuentro del capítulo 2 (transformaciones: rotar, reescalar, resize). No hace falta resolverlo en el mismo encuentro que histogramas o umbrales — funciona mejor justo antes o después de trabajar `skimage.transform`.

## Objetivo

Entender que rotar, reflejar o inclinar una imagen no es un truco de índices: es una multiplicación de matriz. Y que el orden en que aplicamos varias transformaciones cambia el resultado final.

## Al terminar este material van a poder:

1. Explicar qué es una transformación lineal en términos de vectores.
2. Leer una matriz de transformación mirando qué le hace a los vectores de la base.
3. Visualizar cómo una matriz transforma vectores en el plano.
4. Aplicar rotación y shear (inclinación) a una imagen real usando matrices con `scikit-image`.
5. Verificar que el orden de aplicación de las transformaciones cambia el resultado.

## Microglosario

| Término | Significado en lenguaje llano |
|---|---|
| Vector | Una flecha con dirección y magnitud, representada como una lista de números. |
| Transformación lineal | Una función que mueve vectores de lugar, manteniendo las rectas como rectas y el origen fijo. |
| Matriz de transformación | La tabla de números que, multiplicada por un vector, produce el vector transformado. |
| Base estándar | Los vectores más simples posibles — por ejemplo (1,0) y (0,1) — que sirven de referencia para leer una transformación. |
| Shear (inclinación) | Una transformación que empuja un lado de la figura hacia un costado, como inclinar una pila de hojas. |
| Conmutatividad | La propiedad de que el orden de una operación no cambia el resultado. La multiplicación de matrices generalmente NO la cumple. |

## Estirar una hoja de goma

Pensemos una transformación lineal como estirar una hoja de goma que tiene dibujadas un montón de flechas en distintas direcciones. Al estirar la hoja, casi todas las flechas cambian de largo y de dirección. Pero las rectas siguen siendo rectas (ninguna se curva) y el punto central de la hoja no se mueve.

Lo interesante es que **toda transformación de este tipo se puede escribir como una multiplicación de matriz**: la matriz es la "receta" completa de cómo se estira la hoja, y multiplicarla por un vector es aplicarle esa receta a esa flecha en particular.

## 1. Una transformación como multiplicación de matriz

Si tenemos una matriz `A` de 2x2 y un vector `v` de 2x1, el producto `A @ v` nos da un nuevo vector: el resultado de aplicarle la transformación a `v`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

matriz_ejemplo = np.array([
    [2, 0],
    [0, 1],
])

vector_original = np.array([[3], [5]])
vector_transformado = matriz_ejemplo @ vector_original

print("✓ Transformación aplicada.")
print("Vector original:\n", vector_original)
print("Vector transformado:\n", vector_transformado)

### Para pensar

- Miren los números de `matriz_ejemplo`. ¿Qué le pasó a la primera coordenada del vector? ¿Y a la segunda?
- Prueben cambiar el `2` de la matriz por un `0.5`. ¿Qué esperan que pase con el vector transformado?

## 2. Leer una matriz mirando la base estándar

Los vectores más simples del plano son $e_1 = (1, 0)$ y $e_2 = (0, 1)$: apuntan puramente en horizontal y en vertical. Si le aplicamos una transformación a estos dos vectores, las columnas de la matriz resultante son exactamente las columnas de la matriz de transformación. Es decir: **mirando a dónde van a parar $e_1$ y $e_2$, ya sabemos leer toda la matriz**, sin necesitar más ejemplos.

Vamos a armar una función para graficar los vectores originales (en un color) y transformados (en otro), y comparar visualmente. No hace falta entender cada línea de esta función de graficado — lo importante es leer el resultado.

In [ ]:
def graficar_transformacion(matriz, e1, e2):
    color_original = "#129cab"
    color_transformado = "#cc8933"

    _, eje = plt.subplots(figsize=(6, 6))
    eje.set_xticks(np.arange(-4, 5))
    eje.set_yticks(np.arange(-4, 5))
    plt.axis([-4, 4, -4, 4])
    plt.grid(True, alpha=0.3)

    e1_transformado = matriz @ e1
    e2_transformado = matriz @ e2

    plt.quiver([0, 0], [0, 0], [e1[0], e2[0]], [e1[1], e2[1]],
               color=color_original, angles="xy", scale_units="xy", scale=1,
               label="Base original (e1, e2)")
    plt.quiver([0, 0], [0, 0], [e1_transformado[0], e2_transformado[0]], [e1_transformado[1], e2_transformado[1]],
               color=color_transformado, angles="xy", scale_units="xy", scale=1,
               label="Base transformada")

    plt.legend()
    plt.gca().set_aspect("equal")
    plt.show()

print("✓ Función graficar_transformacion creada.")

## 3. Ejemplo: escalado horizontal

Esta matriz estira el eje horizontal al doble, sin tocar el vertical.

In [ ]:
matriz_escalado_horizontal = np.array([
    [2, 0],
    [0, 1],
])

e1 = np.array([[1], [0]])
e2 = np.array([[0], [1]])

graficar_transformacion(matriz_escalado_horizontal, e1, e2)

## 4. Ejemplo: reflexión sobre el eje vertical

Esta matriz invierte el signo de la coordenada horizontal — como reflejar en un espejo parado.

In [ ]:
matriz_reflexion = np.array([
    [-1, 0],
    [0, 1],
])

graficar_transformacion(matriz_reflexion, e1, e2)

### Para pensar

- En la reflexión, ¿qué vector cambió de dirección y cuál se quedó igual?
- ¿Qué matriz esperarían para una reflexión sobre el eje horizontal en vez del vertical?

## 5. Aplicación real: transformar una imagen

Una gran cantidad de gráficos por computadora se arma repitiendo formas simples con distintas transformaciones (rotaciones, reflexiones, inclinaciones). Por ejemplo, muchas ilustraciones de hojas o helechos se generan aplicando la misma transformación una y otra vez sobre una hoja base — en vez de dibujar cada hoja a mano.

Vamos a aplicar dos transformaciones sobre una imagen real: una rotación y un shear (inclinación).

In [ ]:
from skimage import data
from skimage.transform import rotate, AffineTransform, warp

imagen = data.camera()

plt.figure(figsize=(6, 6))
plt.imshow(imagen, cmap="gray")
plt.title("Imagen original")
plt.axis("off")
plt.show()

print("✓ Imagen cargada.")

### Rotación

La matriz de rotación de 90° en sentido horario es exactamente esta:

$$\begin{bmatrix} 0 & 1 \\ -1 & 0 \end{bmatrix}$$

Para aplicarla a una imagen completa (no solo a un vector), usamos la función `rotate` de `skimage.transform` — la misma que van a usar en el capítulo 2. Por dentro, aplica una rotación equivalente a esta multiplicación de matriz sobre las coordenadas de cada píxel.

In [ ]:
matriz_rotacion_90 = np.array([
    [0, 1],
    [-1, 0],
])

print("Matriz de rotación de 90° en sentido horario:\n", matriz_rotacion_90)

imagen_rotada = rotate(imagen, -90)

plt.figure(figsize=(6, 6))
plt.imshow(imagen_rotada, cmap="gray")
plt.title("Imagen rotada 90° en sentido horario")
plt.axis("off")
plt.show()

### Shear (inclinación)

Para el shear usamos `AffineTransform`, que arma la matriz de transformación por nosotros a partir de un ángulo. Podemos pedirle que nos muestre esa matriz con `.params` — por atrás, sigue siendo exactamente el mismo mecanismo que venimos usando.

`warp` necesita la transformación **inversa** (`.inverse`), porque para cada píxel de la imagen de salida necesita saber de qué píxel de la imagen original viene. No hace falta que memoricen ese detalle, solo que sepan que `warp` siempre se usa así.

In [ ]:
transformacion_shear = AffineTransform(shear=0.4)

print("Matriz de la transformación de shear (forma homogénea 3x3):\n", transformacion_shear.params)

imagen_inclinada = warp(imagen, transformacion_shear.inverse)

plt.figure(figsize=(6, 6))
plt.imshow(imagen_inclinada, cmap="gray")
plt.title("Imagen con shear")
plt.axis("off")
plt.show()

### Para pensar

- ¿Qué línea de la imagen original se mantuvo igual después del shear?
- Prueben cambiar `shear=0.4` por un valor negativo, por ejemplo `-0.4`. ¿Hacia qué lado se inclina ahora?

## 6. El orden importa

Ahora apliquemos las dos transformaciones en fila, pero en dos órdenes distintos: primero rotar y después inclinar, y al revés.

In [ ]:
# Orden A: primero rotamos, después inclinamos
imagen_rotada_luego_inclinada = warp(rotate(imagen, -90), transformacion_shear.inverse)

# Orden B: primero inclinamos, después rotamos
imagen_inclinada_luego_rotada = rotate(warp(imagen, transformacion_shear.inverse), -90)

figura, ejes = plt.subplots(1, 2, figsize=(12, 6))
ejes[0].imshow(imagen_rotada_luego_inclinada, cmap="gray")
ejes[0].set_title("Orden A: rotar y después inclinar")
ejes[0].axis("off")

ejes[1].imshow(imagen_inclinada_luego_rotada, cmap="gray")
ejes[1].set_title("Orden B: inclinar y después rotar")
ejes[1].axis("off")

plt.tight_layout()
plt.show()

print("✓ Comparación generada.")

### Para pensar

- ¿Las dos imágenes son iguales? ¿Esperaban ese resultado?
- Piensen en ponerse las medias y los zapatos: ¿da lo mismo el orden? Relacionen esa idea con lo que acaban de ver.

## 7. Confirmando con las matrices

Esto no es casualidad: aplicar una transformación `A` y después una transformación `B` equivale a multiplicar las matrices en el orden `B @ A` (la última transformación aplicada va primero en el producto). Como la multiplicación de matrices generalmente no es conmutativa, el orden cambia el resultado.

In [ ]:
matriz_shear_2x2 = np.array([
    [1, 0.5],
    [0, 1],
])

print("Rotar y después inclinar equivale a (shear @ rotación):\n", matriz_shear_2x2 @ matriz_rotacion_90)
print()
print("Inclinar y después rotar equivale a (rotación @ shear):\n", matriz_rotacion_90 @ matriz_shear_2x2)

### Para pensar

- ¿Las dos matrices resultantes son iguales? ¿El hecho de que sean distintas es consistente con el patrón que vieron en las imágenes (dos resultados diferentes según el orden)?

## Cierre

En este cuaderno vimos que rotar, reflejar o inclinar una imagen es, por dentro, una multiplicación de matriz — y que aplicar varias transformaciones en distinto orden da resultados distintos, porque la multiplicación de matrices no es conmutativa.

La próxima vez que usen `skimage.transform.rotate`, `rescale` o `resize` en el capítulo 2, van a estar usando exactamente esta misma idea, aunque la librería se encargue de armar la matriz por ustedes.

## Actividad de producción

Elijan una imagen de `scikit-image` (por ejemplo `data.astronaut()` convertida a escala de grises, o `data.coffee()`) y:

1. Apliquen una rotación y un shear con un valor de ángulo a elección.
2. Comparen el resultado aplicando las transformaciones en los dos órdenes posibles.
3. Confirmen con las matrices 2x2 que el resultado coincide con lo que predice el álgebra.

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────
#
# 1. Elijan una imagen de scikit-image (conviértanla a escala de grises si hace falta)
# 2. Definan una rotación y un shear con AffineTransform
# 3. Aplíquenlos en los dos órdenes posibles y comparen las imágenes resultantes
# 4. Confirmen el resultado con las matrices 2x2
#
# ─────────────────────────────────────────────────────────────────────────
